<a href="https://colab.research.google.com/github/RolandoLopez16/RegresionLinealHV/blob/main/clima_optimizado_dia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clima histórico por finca y día — versión optimizada

Este notebook toma el archivo `Dataset_Clima.xlsx`, convierte coordenadas sexagesimales a decimales, combina `FECHA` con los turnos y consulta clima histórico tomando como referencia el **día completo**.

Estrategia de rendimiento:

1. Limpieza vectorizada de fecha/hora y coordenadas.
2. Consulta por combinaciones únicas de `latitud + longitud + fecha`, no por cada fila.
3. Fuente principal: `Meteostat Daily`.
4. Respaldo: `Meteostat Hourly` agregado por día.
5. Respaldo opcional: `Open-Meteo` para reducir registros sin dato.

## 1. Instalar librerías

In [ ]:
!pip install meteostat==1.6.8 --no-deps -q
!pip install openpyxl tqdm requests -q

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Importar librerías y configurar entorno

In [ ]:
from pathlib import Path
from datetime import datetime, date, time, timedelta
import pandas as pd
import numpy as np
import re
import requests
from tqdm import tqdm

# Compatibilidad Meteostat con NumPy moderno
if not hasattr(np, "NaN"):
    np.NaN = np.nan

from meteostat import Point, Daily, Hourly

tqdm.pandas()
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

print("Entorno cargado correctamente")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

## 4. Definir rutas y parámetros

In [ ]:
ruta_archivo = Path('/content/drive/MyDrive/Final_Esp_Data/Dataset_Clima.xlsx')
carpeta = Path('/content/drive/MyDrive/Final_Esp_Data')

ruta_salida_excel = carpeta / 'Dataset_Clima_Con_Historico_Dia.xlsx'
ruta_salida_csv = carpeta / 'Dataset_Clima_Con_Historico_Dia.csv'

# True: si Meteostat no trae datos, intenta completar con Open-Meteo.
# False: usa solamente Meteostat.
USAR_OPEN_METEO_FALLBACK = True

# Umbral para clasificar día lluvioso.
# Si la precipitación diaria es mayor que este valor, se marca como Lluvioso.
UMBRAL_LLUVIA_MM = 0.0

print("Archivo origen:", ruta_archivo)
print("Archivo existe:", ruta_archivo.exists())
print("Carpeta salida existe:", carpeta.exists())

## 5. Cargar dataset

In [ ]:
if not ruta_archivo.exists():
    raise FileNotFoundError(f"No existe el archivo: {ruta_archivo}")

df = pd.read_excel(ruta_archivo)
df.columns = df.columns.str.strip()

print("Filas:", len(df))
print("Columnas:", df.columns.tolist())
df.head()

## 6. Validar columnas requeridas

In [ ]:
columnas_requeridas = [
    "Longitud Sexa finca",
    "Latitud Sexa finca",
    "Municipio Finca",
    "HORA INICIO TURNO",
    "HORA FIN TURNO",
    "FECHA"
]

faltantes = [c for c in columnas_requeridas if c not in df.columns]
if faltantes:
    raise ValueError(f"Faltan columnas obligatorias en el archivo: {faltantes}")

print("Columnas obligatorias completas")

## 7. Funciones de limpieza

In [ ]:
def dms_to_decimal(valor):
    """Convierte coordenadas tipo 75°50'28.7''W o 4°10'32.5''N a decimal."""
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().upper()
    texto = texto.replace("''", '"')
    texto = texto.replace("″", '"').replace("”", '"')
    texto = texto.replace("’", "'").replace("′", "'")
    texto = texto.replace(" ", "")

    patron = r"(\d+(?:\.\d+)?)°(\d+(?:\.\d+)?)'(\d+(?:\.\d+)?)\"?([NSEW])"
    match = re.search(patron, texto)

    if match:
        grados = float(match.group(1))
        minutos = float(match.group(2))
        segundos = float(match.group(3))
        direccion = match.group(4)

        decimal = grados + minutos / 60 + segundos / 3600
        if direccion in ["S", "W"]:
            decimal *= -1
        return decimal

    try:
        return float(texto.replace(",", "."))
    except Exception:
        return np.nan


def extraer_hora(valor):
    """Extrae hora desde objetos time, datetime, Timestamp o texto HH:MM:SS."""
    if pd.isna(valor):
        return None

    if isinstance(valor, time):
        return valor

    if isinstance(valor, (datetime, pd.Timestamp)):
        return valor.time()

    texto = str(valor).strip()
    parsed = pd.to_datetime(texto, errors="coerce")
    if pd.isna(parsed):
        return None
    return parsed.time()


def construir_datetime(fecha_valor, hora_valor):
    fecha = pd.to_datetime(fecha_valor, errors="coerce")
    if pd.isna(fecha):
        return pd.NaT

    hora = extraer_hora(hora_valor)
    if hora is None:
        return pd.NaT

    return datetime.combine(fecha.date(), hora)


def estado_lluvia(precipitacion_mm, umbral=UMBRAL_LLUVIA_MM):
    if pd.isna(precipitacion_mm):
        return "Sin dato"
    return "Lluvioso" if float(precipitacion_mm) > umbral else "Seco"

## 8. Preparar coordenadas, fechas y turnos

In [ ]:
# Coordenadas
df["lat_decimal"] = df["Latitud Sexa finca"].apply(dms_to_decimal)
df["lon_decimal"] = df["Longitud Sexa finca"].apply(dms_to_decimal)

# Fecha base del registro
df["fecha_dt"] = pd.to_datetime(df["FECHA"], errors="coerce").dt.normalize()

# Datetime de inicio y fin del turno
df["hora_inicio_turno_dt"] = df.apply(
    lambda row: construir_datetime(row["FECHA"], row["HORA INICIO TURNO"]),
    axis=1
)

df["hora_fin_turno_dt"] = df.apply(
    lambda row: construir_datetime(row["FECHA"], row["HORA FIN TURNO"]),
    axis=1
)

# Si el fin del turno es menor o igual al inicio, se asume que termina al día siguiente.
mask_turno_cruza_dia = (
    df["hora_inicio_turno_dt"].notna() &
    df["hora_fin_turno_dt"].notna() &
    (df["hora_fin_turno_dt"] <= df["hora_inicio_turno_dt"])
)

df.loc[mask_turno_cruza_dia, "hora_fin_turno_dt"] = (
    df.loc[mask_turno_cruza_dia, "hora_fin_turno_dt"] + timedelta(days=1)
)

# Clave de día para consulta climática
df["fecha_clima"] = df["fecha_dt"].dt.date

df[[
    "Longitud Sexa finca", "Latitud Sexa finca", "lat_decimal", "lon_decimal",
    "FECHA", "fecha_dt", "HORA INICIO TURNO", "hora_inicio_turno_dt",
    "HORA FIN TURNO", "hora_fin_turno_dt"
]].head()

## 9. Validación crítica antes de consultar clima

In [ ]:
validacion = {
    "total_registros": len(df),
    "sin_latitud": int(df["lat_decimal"].isna().sum()),
    "sin_longitud": int(df["lon_decimal"].isna().sum()),
    "sin_fecha": int(df["fecha_dt"].isna().sum()),
    "sin_hora_inicio": int(df["hora_inicio_turno_dt"].isna().sum()),
    "sin_hora_fin": int(df["hora_fin_turno_dt"].isna().sum()),
}

for k, v in validacion.items():
    print(f"{k}: {v}")

if validacion["sin_latitud"] or validacion["sin_longitud"] or validacion["sin_fecha"]:
    print("
Revisa las filas con datos base faltantes:")
    display(df[df["lat_decimal"].isna() | df["lon_decimal"].isna() | df["fecha_dt"].isna()].head(20))

## 10. Crear combinaciones únicas para consultar solo una vez por finca/coordenada/día

In [ ]:
base_consulta = df[[
    "lat_decimal",
    "lon_decimal",
    "fecha_dt",
    "Municipio Finca"
] + (["FINCA"] if "FINCA" in df.columns else [])].copy()

base_consulta = base_consulta.dropna(subset=["lat_decimal", "lon_decimal", "fecha_dt"])
base_consulta["lat_key"] = base_consulta["lat_decimal"].round(5)
base_consulta["lon_key"] = base_consulta["lon_decimal"].round(5)
base_consulta["fecha_key"] = base_consulta["fecha_dt"].dt.strftime("%Y-%m-%d")

consultas_unicas = base_consulta.drop_duplicates(subset=["lat_key", "lon_key", "fecha_key"]).reset_index(drop=True)

print("Registros originales:", len(df))
print("Consultas únicas a realizar:", len(consultas_unicas))
print("Reducción de consultas:", f"{(1 - len(consultas_unicas) / max(len(df), 1)):.2%}")
consultas_unicas.head()

## 11. Funciones de consulta climática diaria optimizada

In [ ]:
def consultar_meteostat_daily(lat, lon, fecha_dt):
    """Consulta diaria con Meteostat. Es más rápida y suele tener mejor cobertura que la consulta horaria."""
    fecha = pd.to_datetime(fecha_dt).to_pydatetime().replace(hour=0, minute=0, second=0, microsecond=0)
    punto = Point(float(lat), float(lon))

    data = Daily(punto, fecha, fecha).fetch()
    if data.empty:
        return None

    row = data.iloc[0]
    return {
        "clima_fuente": "Meteostat Daily",
        "clima_temp_promedio_dia_c": row.get("tavg", np.nan),
        "clima_temp_min_dia_c": row.get("tmin", np.nan),
        "clima_temp_max_dia_c": row.get("tmax", np.nan),
        "clima_precipitacion_dia_mm": row.get("prcp", np.nan),
        "clima_viento_promedio_dia_kmh": row.get("wspd", np.nan),
        "clima_presion_promedio_dia_hpa": row.get("pres", np.nan),
        "clima_horas_validas_dia": np.nan,
        "clima_horas_lluvia_dia": np.nan,
        "clima_observacion_dia": "Dato diario obtenido correctamente"
    }


def consultar_meteostat_hourly_agregado(lat, lon, fecha_dt):
    """Respaldo con Meteostat Hourly: consulta todo el día y agrega precipitación, temperatura, viento y presión."""
    inicio = pd.to_datetime(fecha_dt).to_pydatetime().replace(hour=0, minute=0, second=0, microsecond=0)
    fin = inicio + timedelta(days=1)
    punto = Point(float(lat), float(lon))

    data = Hourly(punto, inicio, fin).fetch()
    if data.empty:
        return None

    prcp = data["prcp"] if "prcp" in data.columns else pd.Series(dtype=float)
    horas_validas = int(prcp.notna().sum()) if len(prcp) else 0

    if horas_validas == 0:
        prcp_total = np.nan
        horas_lluvia = np.nan
    else:
        prcp_total = float(prcp.sum(skipna=True))
        horas_lluvia = int((prcp > 0).sum())

    return {
        "clima_fuente": "Meteostat Hourly agregado",
        "clima_temp_promedio_dia_c": data["temp"].mean(skipna=True) if "temp" in data.columns else np.nan,
        "clima_temp_min_dia_c": data["temp"].min(skipna=True) if "temp" in data.columns else np.nan,
        "clima_temp_max_dia_c": data["temp"].max(skipna=True) if "temp" in data.columns else np.nan,
        "clima_precipitacion_dia_mm": prcp_total,
        "clima_viento_promedio_dia_kmh": data["wspd"].mean(skipna=True) if "wspd" in data.columns else np.nan,
        "clima_presion_promedio_dia_hpa": data["pres"].mean(skipna=True) if "pres" in data.columns else np.nan,
        "clima_horas_validas_dia": horas_validas,
        "clima_horas_lluvia_dia": horas_lluvia,
        "clima_observacion_dia": "Dato horario agregado a día obtenido correctamente"
    }


def consultar_open_meteo_daily(lat, lon, fecha_dt):
    """Fallback opcional. Útil cuando Meteostat no tiene datos por estación cercana."""
    fecha = pd.to_datetime(fecha_dt).strftime("%Y-%m-%d")
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": float(lat),
        "longitude": float(lon),
        "start_date": fecha,
        "end_date": fecha,
        "daily": "temperature_2m_mean,temperature_2m_min,temperature_2m_max,precipitation_sum,windspeed_10m_max",
        "timezone": "America/Bogota"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    if "daily" not in data or not data["daily"].get("time"):
        return None

    daily = data["daily"]
    return {
        "clima_fuente": "Open-Meteo Archive",
        "clima_temp_promedio_dia_c": daily.get("temperature_2m_mean", [np.nan])[0],
        "clima_temp_min_dia_c": daily.get("temperature_2m_min", [np.nan])[0],
        "clima_temp_max_dia_c": daily.get("temperature_2m_max", [np.nan])[0],
        "clima_precipitacion_dia_mm": daily.get("precipitation_sum", [np.nan])[0],
        "clima_viento_promedio_dia_kmh": daily.get("windspeed_10m_max", [np.nan])[0],
        "clima_presion_promedio_dia_hpa": np.nan,
        "clima_horas_validas_dia": np.nan,
        "clima_horas_lluvia_dia": np.nan,
        "clima_observacion_dia": "Dato obtenido por fallback Open-Meteo"
    }


def consultar_clima_dia_unico(row):
    lat = row["lat_decimal"]
    lon = row["lon_decimal"]
    fecha = row["fecha_dt"]

    base_error = {
        "clima_fuente": "Sin fuente",
        "clima_temp_promedio_dia_c": np.nan,
        "clima_temp_min_dia_c": np.nan,
        "clima_temp_max_dia_c": np.nan,
        "clima_precipitacion_dia_mm": np.nan,
        "clima_viento_promedio_dia_kmh": np.nan,
        "clima_presion_promedio_dia_hpa": np.nan,
        "clima_horas_validas_dia": 0,
        "clima_horas_lluvia_dia": np.nan,
        "clima_observacion_dia": "Sin dato"
    }

    if pd.isna(lat) or pd.isna(lon) or pd.isna(fecha):
        base_error["clima_observacion_dia"] = "Datos insuficientes"
        return pd.Series(base_error)

    errores = []

    for nombre, funcion in [
        ("Meteostat Daily", consultar_meteostat_daily),
        ("Meteostat Hourly agregado", consultar_meteostat_hourly_agregado),
    ]:
        try:
            resultado = funcion(lat, lon, fecha)
            if resultado is not None:
                prcp = resultado.get("clima_precipitacion_dia_mm", np.nan)
                resultado["clima_estado_dia"] = estado_lluvia(prcp)
                resultado["fue_lluvioso_dia"] = "Sí" if pd.notna(prcp) and prcp > UMBRAL_LLUVIA_MM else ("No" if pd.notna(prcp) else "Sin dato")
                return pd.Series(resultado)
        except Exception as e:
            errores.append(f"{nombre}: {e}")

    if USAR_OPEN_METEO_FALLBACK:
        try:
            resultado = consultar_open_meteo_daily(lat, lon, fecha)
            if resultado is not None:
                prcp = resultado.get("clima_precipitacion_dia_mm", np.nan)
                resultado["clima_estado_dia"] = estado_lluvia(prcp)
                resultado["fue_lluvioso_dia"] = "Sí" if pd.notna(prcp) and prcp > UMBRAL_LLUVIA_MM else ("No" if pd.notna(prcp) else "Sin dato")
                return pd.Series(resultado)
        except Exception as e:
            errores.append(f"Open-Meteo: {e}")

    base_error["clima_estado_dia"] = "Sin dato"
    base_error["fue_lluvioso_dia"] = "Sin dato"
    base_error["clima_observacion_dia"] = "No se encontró dato climático. " + " | ".join(errores[:3])
    return pd.Series(base_error)

## 12. Ejecutar consultas climáticas únicas

In [ ]:
resultados_clima_unicos = consultas_unicas.progress_apply(consultar_clima_dia_unico, axis=1)

clima_unico = pd.concat([
    consultas_unicas[["lat_key", "lon_key", "fecha_key"]].reset_index(drop=True),
    resultados_clima_unicos.reset_index(drop=True)
], axis=1)

print("Consultas climáticas realizadas:", len(clima_unico))
print("Resumen por fuente:")
print(clima_unico["clima_fuente"].value_counts(dropna=False))

clima_unico.head()

## 13. Unir clima al dataset original

In [ ]:
df["lat_key"] = df["lat_decimal"].round(5)
df["lon_key"] = df["lon_decimal"].round(5)
df["fecha_key"] = df["fecha_dt"].dt.strftime("%Y-%m-%d")

df_final = df.merge(
    clima_unico,
    on=["lat_key", "lon_key", "fecha_key"],
    how="left"
)

# Limpieza de columnas técnicas opcional: se conservan porque ayudan a auditoría.
print("Filas originales:", len(df))
print("Filas finales:", len(df_final))

df_final[[
    "Municipio Finca",
    "FINCA" if "FINCA" in df_final.columns else "Municipio Finca",
    "FECHA",
    "HORA INICIO TURNO",
    "HORA FIN TURNO",
    "lat_decimal",
    "lon_decimal",
    "clima_fuente",
    "clima_precipitacion_dia_mm",
    "clima_estado_dia",
    "fue_lluvioso_dia",
    "clima_observacion_dia"
]].head(20)

## 14. Resumen de calidad de datos climáticos

In [ ]:
total = len(df_final)
sin_data = df_final["clima_estado_dia"].fillna("Sin dato").eq("Sin dato").sum()
con_data = total - sin_data

print("Total registros:", total)
print("Con dato climático:", con_data, f"({con_data / total:.2%})")
print("Sin dato climático:", sin_data, f"({sin_data / total:.2%})")

print("
Resumen estado día:")
print(df_final["clima_estado_dia"].value_counts(dropna=False))

print("
Resumen fuente:")
print(df_final["clima_fuente"].value_counts(dropna=False))

print("
Registros sin dato por municipio/finca:")
cols_grupo = ["Municipio Finca"] + (["FINCA"] if "FINCA" in df_final.columns else [])
display(
    df_final[df_final["clima_estado_dia"].fillna("Sin dato").eq("Sin dato")]
    .groupby(cols_grupo, dropna=False)
    .size()
    .reset_index(name="registros_sin_dato")
    .sort_values("registros_sin_dato", ascending=False)
    .head(30)
)

## 15. Guardar archivo final

In [ ]:
carpeta.mkdir(parents=True, exist_ok=True)

df_final.to_excel(ruta_salida_excel, index=False)
df_final.to_csv(ruta_salida_csv, index=False, encoding="utf-8-sig")

print("Archivo Excel generado:")
print(ruta_salida_excel)

print("Archivo CSV generado:")
print(ruta_salida_csv)